# 반려견 identity LoRA 학습 + 귀여운 스타일 생성
런타임 -> 런타임 유형 변경 -> T4 GPU 로 먼저 설정하세요.

## 1. GPU 확인

In [ ]:
!nvidia-smi

Fri Sep  4 02:01:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive 마운트 (세션 끊김 대비, 결과물을 Drive에 저장)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/dog_lora_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/dog_photos', exist_ok=True)
print(f'프로젝트 폴더: {PROJECT_DIR}')
print('dog_photos 폴더에 반려견 사진 5장을 넣어주세요 (Drive에서 직접 업로드하거나 아래 셀 사용).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
프로젝트 폴더: /content/drive/MyDrive/dog_lora_project
dog_photos 폴더에 반려견 사진 5장을 넣어주세요 (Drive에서 직접 업로드하거나 아래 셀 사용).


## 3. 패키지 설치

In [ ]:
!pip install -q diffusers transformers accelerate peft bitsandbytes safetensors

## 4. 공식 DreamBooth LoRA 스크립트 다운로드

In [ ]:
# 라이브러리를 특정 릴리즈로 고정
!pip install -q "diffusers==0.31.0" "peft==0.13.2" "transformers==4.46.3" accelerate bitsandbytes safetensors

In [ ]:
# 스크립트도 같은 태그(v0.31.0)에서 받기 - main이 아님에 주의
!mkdir -p dreambooth_lora_official
!curl -sL https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora.py -o dreambooth_lora_official/train_dreambooth_lora.py
!curl -sL https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/requirements.txt -o dreambooth_lora_official/requirements.txt

In [ ]:
import diffusers, peft, transformers
print(diffusers.__version__, peft.__version__, transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

0.31.0 0.13.2 4.46.3


## 5. 반려견 사진 업로드
Drive에 이미 넣어두셨다면 이 셀은 건너뛰어도 됩니다.

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()  # 강아지 사진 5장을 선택해서 업로드
for fname in uploaded.keys():
    shutil.move(fname, f'{PROJECT_DIR}/dog_photos/{fname}')

print('업로드된 파일:')
!ls {PROJECT_DIR}/dog_photos

## 6. 학습 실행
800 step 기준 T4에서 약 15~20분 소요됩니다.

In [ ]:
# mixed_precision을 안 주면 기본값 fp32로 학습 (LoRA + GradScaler fp16 충돌 회피)
!accelerate launch dreambooth_lora_official/train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="{PROJECT_DIR}/dog_photos" \
  --class_data_dir="{PROJECT_DIR}/class_dog" \
  --output_dir="{PROJECT_DIR}/lora_my_dog" \
  --instance_prompt="a photo of sks dog" \
  --class_prompt="a photo of a dog" \
  --with_prior_preservation \
  --prior_loss_weight=1.0 \
  --num_class_images=50 \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=800 \
  --checkpointing_steps=200

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
model_index.json: 100% 541/541 [00:00<00:00, 3.70MB/s]
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]
preprocessor_config.json: 100% 342/342 [00:00<00:00, 2.01MB/s]

scheduler_config.json:   0% 0.00/308 [00:00<?, ?B/s]

scheduler_config.json: 100% 308/308 [00:00<00:00, 86.8kB/s]

special_tokens_map.json:   0% 0.00/472 [00:00<?, ?B/s]
special_tokens_map.json: 100% 472/472 [00:00<00:00, 63.3kB/s]
tokenizer_config.json: 100% 806/806 [00:00<00:00, 253kB/s]

config.json: 100% 617/617 [00:00<00:00, 5.08MB/s]

vocab.json: 1.06MB [00:00, 40.8MB/s]
merges.txt: 525kB [00:00, 43.3MB/s]

text_encoder/model.safetensors:   0% 0.00/492M [00:00<?, ?B/s]
text_encoder/model.saf

## 7. 학습된 LoRA로 귀여운 스타일 이미지 생성

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_PATH = f"{PROJECT_DIR}/lora_my_dog"  # 과적합 체크 후 checkpoint-XXX 경로로 바꿔도 됨

pipe = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, safety_checker=None
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights(LORA_PATH)

prompt = (
    "a photo of sks dog, chibi style, kawaii, big sparkling eyes, "
    "soft pastel colors, cute illustration, clean lineart, high quality"
)
negative_prompt = (
    "realistic, deformed, extra limbs, blurry, low quality, ugly, text, watermark"
)

generator = torch.Generator(device="cuda").manual_seed(42)
images = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_images_per_prompt=4,
    guidance_scale=7.5,
    num_inference_steps=30,
    generator=generator,
).images

for i, img in enumerate(images):
    save_path = f'{PROJECT_DIR}/cute_dog_{i}.png'
    img.save(save_path)
    print(f'saved -> {save_path}')

images[0]

RuntimeError: Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
Failed to import diffusers.models.autoencoders.autoencoder_kl because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.peft because of the following error (look up to see its traceback):
cannot import name 'disable_input_dtype_casting' from 'peft.helpers' (/usr/local/lib/python3.13/dist-packages/peft/helpers.py)

## 8. (선택) 다른 checkpoint 비교
과적합(배경/포즈가 매번 똑같이 나옴) 되었다면 200/400/600 스텝 지점의 LoRA로 바꿔서 비교하세요.

In [ ]:
!ls {PROJECT_DIR}/lora_my_dog
# 예: LORA_PATH = f"{PROJECT_DIR}/lora_my_dog/checkpoint-400" 로 바꿔서 7번 셀 재실행